# Actual energy runs and a complete renovation assessment

This notebook goes from a Core baseline and Data constructions to two renovations, actual annual engine runs and per-step LCA. Install the integration and examples extras. The default is the real RC engine, with its bundled Gothenburg weather; no EnergyPlus installation is needed. This is an illustrative building, not a measured or calibrated case.

The code in actual_renovation.py explicitly runs the baseline, Data Advanced windows, then the Data Advanced outer insulation. It uses Boverket conservative production, transport and installation-waste values. Window-removal factors and electricity/district-heating climate factors are **declared assumptions** because Data does not supply those values here. The annual energy is calculated by the engine, not invented.

In [ ]:
from pathlib import Path

from actual_renovation import run_case
from teaching_case import show_table

result = run_case(output=Path("../outputs/actual-renovation"), engine="epsm_rc")
report = result["report"]
assert report.complete and len(report.stages) == 2
show_table(report)

Operational energy comes from actual annual outputs: purchased district heat plus ventilation-fan electricity. Energy uses the reached state and the same 30-year period per row. Added and removed quantities are taken from Core. Their material factors come from Data or the explicitly labelled removal assumptions.

In [ ]:
for run in result["energy_runs"]:
    print("Step", run["step"], "annual delivered kWh:", run["annual_delivered_kwh"])
    print("By carrier:", run["delivered_by_carrier"])
    assert run["annual_delivered_kwh"] > 0
print("Materials removed:", report.quantities(action="removed", by_material=True))
print("Insulation added:", report.quantities(action="added", step=2, by_material=True))
report.plot()

Every execution creates a new output folder and preserves weather, actual engine outputs, EPSM evaluations, supplied annual energy, the LCA report and comparison CSV. Saving does not overwrite previous runs. Check the source descriptions before drawing research conclusions; a successful engine run does not prove building accuracy.

In [ ]:
from epsm_lca import RenovationResult

assert RenovationResult.load(result["report_path"]) == report
print("Report:", result["report_path"])
print("All retained artifacts:", result["directory"])